# Course 1, Module 2 — Exercises 1 & 2 (rebuilt locally)

Spec: `SPEC.md` in this folder.

## Setup

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

# usecwd=True: in a notebook there is no __file__, so search upward from the working dir.
load_dotenv(find_dotenv(usecwd=True, raise_error_if_not_found=True))

REQUIRED_ENV = ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "OPENAI_MODEL"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing or empty in .env: {', '.join(missing)}")  # names only, never values

CLAUDE_MODEL = "claude-haiku-5-5"  # both exercises; `temperature` is never passed (SPEC §5)
OPENAI_MODEL = os.environ["OPENAI_MODEL"]
MAX_TOKENS = 1024

GOLDEN_PATH = Path.cwd() / "data" / "golden_movies.json"
if not GOLDEN_PATH.exists():
    raise FileNotFoundError(f"{GOLDEN_PATH} not found; run the notebook from course_01/m2_langchain_basics/")
golden = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))

# Sanity checks on the golden file itself, so a typo fails here rather than skewing scores later.
assert len(golden) == 21, f"expected 21 golden rows, got {len(golden)}"
assert [r["id"] for r in golden] == list(range(1, 22)), "golden ids must be 1..21 in order"
counts = pd.Series([r["category"] for r in golden]).value_counts().to_dict()
assert counts == {"scored": 18, "ambiguous": 2, "fake": 1}, f"unexpected category counts: {counts}"

print(f"Claude: {CLAUDE_MODEL} | OpenAI: {OPENAI_MODEL}")
print(f"Golden rows: {len(golden)} {counts}")

## Exercise 1 pre-check: does `OPENAI_MODEL` accept `temperature=0`? (1 API call)

In [ ]:
import openai
from langchain_openai import ChatOpenAI

gpt_check_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_retries=0, timeout=60)

# Offline check first: confirm temperature is really in the request (it was silently relocated on the Anthropic side).
payload = gpt_check_llm._get_request_payload([("human", "ping")])
assert payload.get("temperature") == 0, f"temperature not in request: { {k: v for k, v in payload.items() if k != 'messages'} }"
print("temperature in request:", payload["temperature"])

# One real call. A 400 that mentions temperature = "not testable on current models" (SPEC §8).
# Any other error is raised, not recorded.
try:
    check_msg = gpt_check_llm.invoke("Reply with exactly the word: ok")
except openai.BadRequestError as e:
    if "temperature" not in str(e).lower():
        raise
    GPT_TEMPERATURE_SUPPORTED = False
    print("GPT rejected temperature=0 -> temperature section: not testable on current models")
    print("error:", e)
else:
    GPT_TEMPERATURE_SUPPORTED = True
    print("GPT accepted temperature=0 -> GPT temperature section will run")
    print("model:        ", check_msg.response_metadata.get("model_name"))
    print("finish_reason:", check_msg.response_metadata.get("finish_reason"))
    print("text:         ", repr(check_msg.text))

## Exercise 2 helpers: schema check and scoring (no API calls)

In [ ]:
import re

EXPECTED_KEYS = {"title", "director", "year", "genre"}
YEAR_MIN, YEAR_MAX = 1888, 2026


def _is_real_int(x) -> bool:
    # bool is a subclass of int in Python (True == 1), so exclude it explicitly.
    return isinstance(x, int) and not isinstance(x, bool)


def validate_movie(d) -> tuple[bool, str]:
    """Schema check (SPEC §4, §6). Returns (is_valid, reason). Collects every problem, not just the first."""
    if not isinstance(d, dict):
        return False, f"not a dict (got {type(d).__name__})"

    problems = []
    keys = set(d)
    if keys - EXPECTED_KEYS:
        problems.append(f"extra keys: {sorted(keys - EXPECTED_KEYS)}")
    if EXPECTED_KEYS - keys:
        problems.append(f"missing keys: {sorted(EXPECTED_KEYS - keys)}")

    year = d.get("year")
    if year is not None:
        if not _is_real_int(year):
            problems.append(f"year must be int or null, got {type(year).__name__} {year!r}")
        elif not (YEAR_MIN <= year <= YEAR_MAX):
            problems.append(f"year {year} outside {YEAR_MIN}-{YEAR_MAX}")

    for field in ("title", "director", "genre"):
        value = d.get(field)
        if value is not None:
            if not isinstance(value, str):
                problems.append(f"{field} must be str or null, got {type(value).__name__}")
            elif not value.strip():
                problems.append(f"{field} is an empty string")

    return (not problems), ("; ".join(problems) if problems else "ok")


def score_movie(d: dict, golden_row: dict) -> dict:
    """Correctness (SPEC §7 matching rules). Only for rows with category 'scored'."""
    if golden_row["category"] != "scored":
        # Ambiguous/fake rows have no expected answer; scoring them would silently produce False.
        raise ValueError(f"row {golden_row['id']} is '{golden_row['category']}', not scored")

    director = d.get("director")
    surname = golden_row["expected_director_surname"]
    # Whole-word match with an optional plural "s": \b stops "Bong" matching inside "Bongo",
    # and "s?" lets "The Wachowskis" match "Wachowski". re.escape guards odd characters.
    surname_pattern = re.compile(rf"\b{re.escape(surname)}s?\b", re.IGNORECASE)
    director_ok = isinstance(director, str) and surname_pattern.search(director) is not None

    year = d.get("year")
    year_ok = _is_real_int(year) and year == golden_row["expected_year"]  # "1999" != 1999 on purpose

    return {"director_ok": director_ok, "year_ok": year_ok}

### Offline tests (SPEC §7)

In [ ]:
VALID = {"title": "X", "director": "Y", "year": 1999, "genre": "Z"}
MILLER_ROW = {"id": 12, "category": "scored", "expected_director_surname": "Miller", "expected_year": 2015}
BONG_ROW = {"id": 3, "category": "scored", "expected_director_surname": "Bong", "expected_year": 2019}
WACHOWSKI_ROW = {"id": 1, "category": "scored", "expected_director_surname": "Wachowski", "expected_year": 1999}

validate_cases = [
    # (description, input, expected_valid)
    ("control: valid 4-key dict", VALID, True),
    ("year is a string", {**VALID, "year": "1999"}, False),
    ("year is a bool", {**VALID, "year": True}, False),
    ("5 keys", {**VALID, "rating": 9}, False),
    ("year 1850", {**VALID, "year": 1850}, False),
]
for desc, d, expected in validate_cases:
    ok, reason = validate_movie(d)
    assert ok is expected, f"validate_movie [{desc}]: expected {expected}, got {ok} ({reason})"
    print(f"PASS validate: {desc:<28} -> valid={ok} ({reason})")

score_cases = [
    # (description, director, golden_row, expected_director_ok)
    ("George Miller vs Miller", "George Miller", MILLER_ROW, True),
    # KNOWN FALSE PASS of the surname rule: Frank Miller did not direct Mad Max: Fury Road,
    # but "Miller" appears as a whole word, so director_ok is True. Kept to document the limitation (SPEC §10).
    ("Frank Miller vs Miller", "Frank Miller", MILLER_ROW, True),
    ("Bongo Smith vs Bong", "Bongo Smith", BONG_ROW, False),
    ("The Wachowskis vs Wachowski", "The Wachowskis", WACHOWSKI_ROW, True),
]
for desc, director, row, expected in score_cases:
    result = score_movie({**VALID, "director": director, "year": row["expected_year"]}, row)
    assert result["director_ok"] is expected, f"score_movie [{desc}]: expected {expected}, got {result}"
    print(f"PASS score:    {desc:<28} -> {result}")

raise_cases = [
    {"id": 20, "category": "fake", "expected_director_surname": None, "expected_year": None},
    {"id": 18, "category": "ambiguous", "expected_director_surname": None, "expected_year": None},
]
for row in raise_cases:
    try:
        score_movie(VALID, row)
    except ValueError as e:
        print(f"PASS raises:   {row['category']:<28} -> ValueError({e})")
    else:
        raise AssertionError(f"score_movie should raise ValueError for a '{row['category']}' row")

print(f"\nAll {len(validate_cases) + len(score_cases) + len(raise_cases)} offline tests passed.")

## Exercise 2: extraction (prompt + JsonOutputParser)

In [ ]:
import time

import anthropic
from langchain_anthropic import ChatAnthropic
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# Transient errors worth one retry (SPEC §8). Anything else (bad key, bad model name, 400) is raised, not counted.
# APIConnectionError includes APITimeoutError; InternalServerError covers 5xx including OverloadedError.
RETRYABLE_ERRORS = (
    anthropic.APIConnectionError,
    anthropic.RateLimitError,
    anthropic.InternalServerError,
    anthropic.RequestTimeoutError,
)
RETRY_WAIT_SECONDS = 2  # short pause so an immediate retry doesn't hit the same rate limit


def call_with_retry(fn):
    """Call fn(). On a retryable API error, wait and retry once.
    Returns (result, None, attempts) on success or (None, error_message, attempts) after the retry fails."""
    error = None
    for attempt in (1, 2):
        try:
            return fn(), None, attempt
        except RETRYABLE_ERRORS as e:
            error = f"{type(e).__name__}: {e}"
            if attempt == 1:
                print(f"  API error, retrying once: {error}")
                time.sleep(RETRY_WAIT_SECONDS)
    return None, error, 2


class Movie(BaseModel):
    """Used for the parser's format instructions. Validation is done by validate_movie, not pydantic."""
    title: str | None = Field(description="Title of the film")
    director: str | None = Field(description="Director name(s), or null if unknown")
    year: int | None = Field(description="Release year as an integer, or null if unknown")
    genre: str | None = Field(description="Main genre, or null if unknown")


movie_parser = JsonOutputParser(pydantic_object=Movie)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You extract facts about films. If you are not sure of a field's value, or you do not "
     "recognise the film, use null for that field. Never guess or invent values."),
    ("human", "Movie: {movie_name}\n\n{format_instructions}"),
]).partial(format_instructions=movie_parser.get_format_instructions())

extraction_llm = ChatAnthropic(
    model=CLAUDE_MODEL,  # no temperature: Haiku 5.5 rejects it (SPEC §5)
    max_tokens=MAX_TOKENS,
    max_retries=0,  # the SDK would otherwise retry 2x silently; call_with_retry does exactly one, visibly
    timeout=60,
)

# The chain returns the AIMessage (not just text) so stop_reason is available for error analysis.
# JSON parsing happens outside the chain, so the raw text is kept even when parsing fails (SPEC §8).
extraction_chain = extraction_prompt | extraction_llm


def _strict_parse(raw: str):
    """Plain json.loads: no fence stripping, no repair. Returns (dict | None, error | None)."""
    try:
        value = json.loads(raw)
    except json.JSONDecodeError as e:
        return None, f"JSONDecodeError: {e}"
    if not isinstance(value, dict):
        return None, f"parsed to {type(value).__name__}, not a dict"
    return value, None


def _lenient_only_reason(raw: str, stop_reason: str | None, strict_error: str) -> str:
    """Why did a row parse leniently but not strictly? (SPEC §6: must be shown for error analysis.)"""
    if stop_reason == "max_tokens":
        return "truncated at max_tokens (JSON repaired by parser)"
    if raw.strip().startswith("```"):
        return "wrapped in markdown code fences"
    return f"other: {strict_error}"


def extract_movie(movie_name: str) -> dict:
    """SPEC §4. One row of Ex 2 results. status: ok | parse_error | api_error (ok = lenient parse gave a dict)."""
    row = {"movie": movie_name, "raw": None, "stop_reason": None, "parsed": None,
           "strict_parse_ok": False, "lenient_parse_ok": False, "lenient_only_reason": None,
           "status": None, "error": None, "attempts": None}

    msg, api_error, row["attempts"] = call_with_retry(lambda: extraction_chain.invoke({"movie_name": movie_name}))
    if api_error:
        row.update(status="api_error", error=api_error)
        return row

    raw = msg.text
    row.update(raw=raw, stop_reason=msg.response_metadata.get("stop_reason"))

    _, strict_error = _strict_parse(raw)
    row["strict_parse_ok"] = strict_error is None

    try:
        parsed = movie_parser.parse(raw)
    except OutputParserException as e:  # no retry on parse failure (SPEC §8)
        row.update(status="parse_error", error=str(e))
        return row
    if not isinstance(parsed, dict):  # parse rate means "turns into a dict" (SPEC §6)
        row.update(status="parse_error", error=f"parsed to {type(parsed).__name__}, not a dict")
        return row

    row.update(parsed=parsed, lenient_parse_ok=True, status="ok")
    if not row["strict_parse_ok"]:
        row["lenient_only_reason"] = _lenient_only_reason(raw, row["stop_reason"], strict_error)
    return row

In [ ]:
# Single smoke-test call (1 API call). The 21-movie loop comes later.
smoke = extract_movie("The Matrix")
print("status:          ", smoke["status"], f"(attempts: {smoke['attempts']})")
print("error:           ", smoke["error"])
print("stop_reason:     ", smoke["stop_reason"])
print("strict parse:    ", smoke["strict_parse_ok"])
print("lenient parse:   ", smoke["lenient_parse_ok"])
print("lenient-only why:", smoke["lenient_only_reason"])
print("raw text:\n" + str(smoke["raw"]))
print("parsed:          ", smoke["parsed"])
if smoke["status"] == "ok":
    print("validate:        ", validate_movie(smoke["parsed"]))